# Train audio and text teachers — DAIC-WOZ
**Version 1: frozen encoders + participant-level Keras classifiers.**

Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order. GPU is used for frozen feature extraction; the small Keras heads train on CPU to avoid competing framework GPU allocations.

Audio: `facebook/wav2vec2-base-960h` hidden states from participant-only speech at 16 kHz. Text: `sentence-transformers/all-MiniLM-L6-v2` participant responses. Encoder weights are frozen, not fine-tuned for depression. These are reproducible starting teachers, not claimed state-of-the-art models or a reproduction of Huang.

Official training and development splits are used. Test IDs are checked for overlap, but test audio, transcripts and labels are not used for feature extraction, fitting, early stopping or scoring. `full_test_split.csv` is not read.

All persistent outputs stay under `/content/drive/MyDrive/DAIC_WOZ/experiments`. No dataset contents are sent to GitHub or inference APIs. Hugging Face is used only to download encoder weights. Keep notebook outputs cleared when saving to GitHub.

Sources: [Wav2Vec2 model card](https://huggingface.co/facebook/wav2vec2-base-960h), [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).

## 1. Install dependencies
Run before importing ML libraries. If Colab requests a runtime restart after installation, restart and continue from the next cell. Existing compatible packages are reused; exact runtime versions are recorded with each run.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.44,<5', 'torch>=2.6', 'tensorflow>=2.16',
    'soundfile>=0.12', 'scipy', 'scikit-learn', 'pandas', 'tqdm'])


## 2. Configuration
The first run extracts features for 142 participants. Audio extraction is the expensive step; each participant is cached separately. Rerunning resumes completed caches. If a raw file is missing, the notebook checks participant-named ZIPs under DATA_ROOT and recovers only the required audio/transcript into DATA_ROOT/raw_recovered. Missing or duplicate files stop training; participants are never silently dropped.

Up to 128 audio windows and 128 text chunks are sampled uniformly across each interview, independent of labels. Audio windows are at most 10 seconds; transcript redactions are excluded. Separate modality features are pooled using mean and standard deviation. This loses temporal order and is a baseline design choice to report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, re, json, hashlib, gc, math, time, platform
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd

DATA_ROOT = Path('/content/drive/MyDrive/DAIC_WOZ')
RUN_ROOT = DATA_ROOT / 'experiments'
assert DATA_ROOT.is_dir(), f'Dataset folder not found: {DATA_ROOT}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
SEEDS = [103]  # Later use [103, 104, 105, 106, 107]; report all runs.
DO_OOF = True  # Training-only 5-fold out-of-fold predictions for later KD reliability.
N_FOLDS = 5
MAX_EPOCHS = 100
PATIENCE = 12
BATCH_SIZE = 16
FEATURE_CFG = dict(pipeline_version=1, max_segments=128, audio_seconds=10.0,
                   min_audio_seconds=0.5, sample_rate=16000, text_tokens=254,
                   audio_model='facebook/wav2vec2-base-960h',
                   text_model='sentence-transformers/all-MiniLM-L6-v2')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['USE_TF'] = '0'  # Transformers uses PyTorch; classifier heads use TensorFlow later.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
print('Output folder:', RUN_ROOT)


## 3. Find files and validate participant splits
Only filenames ending in `<ID>_AUDIO.wav` and `<ID>_TRANSCRIPT.csv` or `.txt` are accepted. Duplicate matches or missing files stop the run rather than silently dropping participants. If this fails, share the error and folder listing, not transcript contents.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def save_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False))
    tmp.replace(path)

def read_split(filename, labelled):
    path = DATA_ROOT / 'metadata' / filename
    df = pd.read_csv(path); df.columns = df.columns.str.strip().str.lower()
    assert 'participant_id' in df, f'Missing participant_id in {filename}'
    ids = pd.to_numeric(df['participant_id'], errors='raise')
    assert ids.notna().all() and (ids % 1 == 0).all(), 'Invalid participant IDs'
    assert not ids.duplicated().any() and len(ids), 'Empty or duplicate split IDs'
    result = pd.DataFrame({'participant_id': ids.astype(int)})
    if labelled:
        assert 'phq8_binary' in df, f'Missing labels in {filename}'
        labels = pd.to_numeric(df['phq8_binary'], errors='raise')
        assert labels.notna().all() and labels.isin([0, 1]).all(), 'Invalid binary labels'
        assert labels.nunique() == 2, 'Both classes required'
        result['label'] = labels.astype(int)
    return result, sha256_file(path)

train, train_hash = read_split('train_split_Depression_AVEC2017.csv', True)
dev, dev_hash = read_split('dev_split_Depression_AVEC2017.csv', True)
test_ids, test_hash = read_split('test_split_Depression_AVEC2017.csv', False)
sets = [set(d.participant_id) for d in (train, dev, test_ids)]
assert all(not sets[i] & sets[j] for i in range(3) for j in range(i+1, 3)), 'Participant leakage'
manifest = pd.concat([train.assign(split='train'), dev.assign(split='dev')], ignore_index=True)

import zipfile
import shutil

def recover_from_zip(pid, kind, archives, index, root):
    """Recover exactly one missing raw file from a participant-named ZIP."""
    if index.get(pid): return
    pattern = rf'{pid}_AUDIO\.wav' if kind == 'audio' else rf'{pid}_TRANSCRIPT\.(?:csv|txt)'
    matches = []
    for archive in archives:
        if not re.search(rf'(?<!\d){pid}(?!\d)', archive.stem): continue
        try:
            with zipfile.ZipFile(archive) as z:
                for entry in z.infolist():
                    basename = entry.filename.replace('\\', '/').rsplit('/', 1)[-1]
                    if not entry.is_dir() and re.fullmatch(pattern, basename, re.I):
                        matches.append((archive, entry.filename, basename))
        except zipfile.BadZipFile:
            raise ValueError(f'Invalid ZIP for participant {pid}: {archive}')
    if len(matches) > 1:
        raise ValueError(f'{pid}: multiple archived {kind} matches. Resolve duplicate ZIPs: '
                         + '; '.join(str(a) for a, _, _ in matches))
    if not matches: return
    archive, member, basename = matches[0]
    destination = root / 'raw_recovered' / f'{pid}_P' / basename
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        raise FileExistsError(f'Refusing to overwrite: {destination}')
    temporary = destination.with_suffix(destination.suffix + '.partial')
    try:
        # Copy to a controlled basename; never extract archive paths or other entries.
        with zipfile.ZipFile(archive) as z, z.open(member) as src, open(temporary, 'wb') as dst:
            shutil.copyfileobj(src, dst)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    index[pid] = [destination]
    print(f'Recovered {pid} {kind} from {archive.name}')

audio_index, text_index, archives = {}, {}, []
for base, dirs, files in os.walk(DATA_ROOT):
    dirs[:] = [d for d in dirs if d not in {'experiments', 'processed', '.git', '__pycache__'}]
    for name in files:
        path = Path(base) / name
        a = re.fullmatch(r'(\d+)_AUDIO\.wav', name, re.I)
        t = re.fullmatch(r'(\d+)_TRANSCRIPT\.(?:csv|txt)', name, re.I)
        if a: audio_index.setdefault(int(a.group(1)), []).append(path)
        if t: text_index.setdefault(int(t.group(1)), []).append(path)
        if path.suffix.lower() == '.zip': archives.append(path)

for pid in manifest.participant_id:
    for kind, index in [('audio', audio_index), ('transcript', text_index)]:
        recover_from_zip(int(pid), kind, archives, index, DATA_ROOT)

problems = []
for row in manifest.itertuples():
    pid = int(row.participant_id)
    for kind, index in [('audio', audio_index), ('transcript', text_index)]:
        paths = index.get(pid, [])
        if len(paths) != 1:
            detail = '; '.join(str(p.relative_to(DATA_ROOT)) for p in paths)
            problems.append(f'{pid} ({row.split}): {kind} matches={len(paths)} {detail}')
if problems:
    print('Raw search root:', DATA_ROOT)
    print('Participant ZIP files found:', len(archives))
    print('\n'.join(problems))
    raise FileNotFoundError(
        'Raw file coverage is incomplete. Restore missing participant files from your licensed '
        'DAIC-WOZ download, or place the participant ZIP (e.g. 440_P.zip) under DATA_ROOT and rerun. '
        'Check renamed files and resolve duplicates shown above. No participants were dropped; '
        'no model training has started. Experiments and processed folders are excluded from raw discovery.'
    )

manifest['audio_path'] = [str(audio_index[p][0]) for p in manifest.participant_id]
manifest['transcript_path'] = [str(text_index[p][0]) for p in manifest.participant_id]
print(manifest.groupby('split').label.agg(['count', 'sum']).rename(columns={'sum':'depressed'}))
print('Split overlap: none. Test participants reserved:', len(test_ids))


## 4. Read participant turns and prepare reproducible caches
Only speaker `Participant` is retained. Invalid timestamps fail explicitly. Redacted/empty responses are excluded. PHQ questionnaire items, gender and interviewer text are never model inputs.

In [ ]:
def read_turns(path):
    df = pd.read_csv(path, sep='\t')
    df.columns = df.columns.str.strip().str.lower()
    if not {'start_time', 'stop_time', 'speaker', 'value'} <= set(df.columns):
        df = pd.read_csv(path, sep=None, engine='python')
        df.columns = df.columns.str.strip().str.lower()
    required = {'start_time', 'stop_time', 'speaker', 'value'}
    assert required <= set(df.columns), f'Unexpected transcript columns: {list(df.columns)}'
    df = df.loc[df.speaker.astype(str).str.strip().str.lower().eq('participant')].copy()
    df['value'] = df['value'].fillna('').astype(str).str.strip()
    df = df.loc[df.value.ne('') & ~df.value.str.contains('scrubbed|redacted', case=False, regex=True)]
    for col in ['start_time', 'stop_time']:
        df[col] = pd.to_numeric(df[col], errors='raise')
    assert len(df), f'No usable participant turns in {Path(path).name}'
    assert np.isfinite(df[['start_time', 'stop_time']].to_numpy()).all(), 'Nonfinite timestamps'
    assert ((df.start_time >= 0) & (df.stop_time > df.start_time)).all(), 'Invalid timestamps'
    return df.sort_values('start_time').reset_index(drop=True)

def uniform_subset(items, limit):
    if len(items) <= limit: return items
    return [items[i] for i in np.linspace(0, len(items)-1, limit, dtype=int)]

def audio_windows(turns, seconds, minimum):
    windows = []
    for row in turns.itertuples():
        start, stop = float(row.start_time), float(row.stop_time)
        while start < stop:
            end = min(start + seconds, stop)
            if end - start >= minimum: windows.append((start, end))
            start = end
    assert windows, 'No sufficiently long participant speech windows'
    return uniform_subset(windows, FEATURE_CFG['max_segments'])

def pool_segments(features):
    x = np.asarray(features, dtype=np.float32)
    assert x.ndim == 2 and len(x) and np.isfinite(x).all(), 'Invalid encoder features'
    return np.concatenate([x.mean(axis=0), x.std(axis=0)]).astype(np.float32)

from huggingface_hub import model_info
# Resolve immutable model revisions once; retries reuse the recorded revisions.
revision_path = RUN_ROOT / 'features' / 'teacher_v1_encoder_revisions.json'
if revision_path.exists():
    revisions = json.loads(revision_path.read_text())
else:
    revisions = {FEATURE_CFG[k]: model_info(FEATURE_CFG[k]).sha for k in ['audio_model', 'text_model']}
    save_json(revision_path, revisions)
assert all(FEATURE_CFG[k] in revisions for k in ['audio_model', 'text_model']), 'Encoder config changed: use a new revision file'
cache_config = dict(FEATURE_CFG, revisions=revisions,
                    packages={p: version(p) for p in ['transformers', 'torch', 'numpy', 'scipy', 'soundfile']})
cache_key = hashlib.sha256(json.dumps(cache_config, sort_keys=True).encode()).hexdigest()[:16]
CACHE_ROOT = RUN_ROOT / 'features' / ('teacher_v1_' + cache_key)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
save_json(CACHE_ROOT / 'config.json', cache_config)
turns_by_id = {int(r.participant_id): read_turns(r.transcript_path) for r in manifest.itertuples()}
print('Usable participant transcripts:', len(turns_by_id))
print('Cache:', CACHE_ROOT)

def source_signature(row):
    # Full content hashes prevent reuse after source files are replaced.
    return {'audio_sha256': sha256_file(row.audio_path),
            'transcript_sha256': sha256_file(row.transcript_path)}

from tqdm.auto import tqdm
signatures = {int(r.participant_id): source_signature(r)
              for r in tqdm(list(manifest.itertuples()), desc='Fingerprint sources')}

def cache_file(pid, modality): return CACHE_ROOT / f'{int(pid)}_{modality}.npz'

def cached(pid, modality):
    path = cache_file(pid, modality)
    if not path.exists(): return False
    try:
        with np.load(path, allow_pickle=False) as data:
            valid = json.loads(str(data['signature'].item())) == signatures[int(pid)]
            return valid and np.isfinite(data['pooled']).all() and np.isfinite(data['segments']).all()
    except (ValueError, OSError, KeyError): return False

def save_features(pid, modality, features):
    path = cache_file(pid, modality); tmp = path.with_suffix('.tmp')
    with open(tmp, 'wb') as f:
        np.savez_compressed(f, segments=np.asarray(features, dtype=np.float32),
                            pooled=pool_segments(features),
                            signature=json.dumps(signatures[int(pid)], sort_keys=True))
    tmp.replace(path)


## 5. Extract frozen audio features
Wav2Vec2 receives one unpadded window at a time. Audio is resampled to 16 kHz and hidden states are averaged across valid frames. Input is participant speech selected using transcript timestamps. If audio is silent or timestamps exceed its duration, extraction stops with the participant ID for diagnosis.

In [ ]:
import torch
import soundfile as sf
from scipy.signal import resample_poly
from transformers import AutoFeatureExtractor, AutoTokenizer, AutoModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Feature extraction device:', device)
if device == 'cpu': print('CPU extraction can be slow. Select a GPU runtime for this stage.')
pending = [r for r in manifest.itertuples() if not cached(r.participant_id, 'audio')]
if pending:
    name = FEATURE_CFG['audio_model']; revision = revisions[name]
    audio_processor = AutoFeatureExtractor.from_pretrained(name, revision=revision)
    audio_encoder = AutoModel.from_pretrained(name, revision=revision, use_safetensors=True).to(device).eval()
    audio_encoder.requires_grad_(False)
    for row in tqdm(pending, desc='Audio participants'):
        pid = int(row.participant_id); features = []
        windows = audio_windows(turns_by_id[pid], FEATURE_CFG['audio_seconds'], FEATURE_CFG['min_audio_seconds'])
        with sf.SoundFile(row.audio_path) as wav:
            rate = wav.samplerate; duration = len(wav) / rate
            assert turns_by_id[pid].stop_time.max() <= duration + 0.1, f'{pid}: transcript exceeds audio duration'
            for start, end in windows:
                wav.seek(min(round(start * rate), len(wav)))
                x = wav.read(max(1, round((end-start) * rate)), dtype='float32', always_2d=True).mean(axis=1)
                assert len(x) and np.isfinite(x).all(), f'{pid}: empty/nonfinite audio'
                if rate != FEATURE_CFG['sample_rate']:
                    divisor = math.gcd(rate, FEATURE_CFG['sample_rate'])
                    x = resample_poly(x, FEATURE_CFG['sample_rate']//divisor, rate//divisor).astype(np.float32)
                if len(x) < FEATURE_CFG['min_audio_seconds'] * FEATURE_CFG['sample_rate']: continue
                if np.max(np.abs(x)) < 1e-7: continue
                inputs = audio_processor(x, sampling_rate=FEATURE_CFG['sample_rate'], return_tensors='pt')
                with torch.inference_mode():
                    h = audio_encoder(input_values=inputs.input_values.to(device)).last_hidden_state
                    features.append(h.mean(dim=1).squeeze(0).cpu().numpy())
        assert features, f'{pid}: no nonsilent usable audio windows'
        save_features(pid, 'audio', features)
    del audio_encoder, audio_processor, inputs, h
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Audio caches ready:', sum(cached(p, 'audio') for p in manifest.participant_id))


## 6. Extract frozen text features
Long participant responses are divided into token chunks rather than silently truncated. MiniLM uses attention-mask-aware mean pooling and L2 normalization, following its model card. The interview representation concatenates mean and standard deviation across retained chunks.

In [ ]:
pending = [r for r in manifest.itertuples() if not cached(r.participant_id, 'text')]
if pending:
    name = FEATURE_CFG['text_model']; revision = revisions[name]
    text_tokenizer = AutoTokenizer.from_pretrained(name, revision=revision)
    text_encoder = AutoModel.from_pretrained(name, revision=revision, use_safetensors=True).to(device).eval()
    text_encoder.requires_grad_(False)
    for row in tqdm(pending, desc='Text participants'):
        pid = int(row.participant_id); chunks = []
        for response in turns_by_id[pid]['value']:
            ids = text_tokenizer.encode(response, add_special_tokens=False)
            for start in range(0, len(ids), FEATURE_CFG['text_tokens']):
                tokens = text_tokenizer.build_inputs_with_special_tokens(ids[start:start+FEATURE_CFG['text_tokens']])
                chunks.append({'input_ids': tokens, 'attention_mask': [1]*len(tokens)})
        chunks = uniform_subset(chunks, FEATURE_CFG['max_segments'])
        assert chunks, f'{pid}: no text tokens'
        features = []
        for start in range(0, len(chunks), 16):
            batch = text_tokenizer.pad(chunks[start:start+16], padding=True, return_tensors='pt')
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.inference_mode():
                h = text_encoder(**batch).last_hidden_state
                mask = batch['attention_mask'].unsqueeze(-1)
                emb = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                emb = torch.nn.functional.normalize(emb, p=2, dim=1)
                features.extend(emb.cpu().numpy())
        save_features(pid, 'text', features)
    del text_encoder, text_tokenizer, batch, h, emb
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Text caches ready:', sum(cached(p, 'text') for p in manifest.participant_id))


## 7. Train participant-level classifiers
Each participant contributes one training example. Scaling is fit on training participants only. The heads use L2 regularization, dropout, training-only class weights and early stopping on development cross-entropy. The decision threshold is fixed at 0.5. Development scores are model-selection results, not an untouched test estimate.

Optional OOF stage (enabled by default): five folds of the **107 training participants only**. Within each outer training fold, an inner holdout selects the epoch count; the model is then refit on the full outer training fold for that count. The outer held-out participant labels never select epochs or fit the scaler. OOF probabilities are saved for later reliability modelling; they are not yet calibrated.

Checkpoints and predictions contain private participant information and remain in Drive. Every execution gets a new run directory; feature caches are reused.

In [ ]:
import tensorflow as tf
# Small heads run on CPU, leaving the GPU exclusively to encoder extraction.
tf.config.set_visible_devices([], 'GPU')
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (f1_score, balanced_accuracy_score, roc_auc_score,
                             average_precision_score, log_loss, confusion_matrix, brier_score_loss)
from scipy.special import expit

def build_head(input_dim, seed):
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-3)),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-3)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
                  loss=tf.keras.losses.BinaryCrossentropy(from_logits=True))
    return model

def class_weights(y):
    counts = np.bincount(np.asarray(y, dtype=int), minlength=2)
    assert (counts > 0).all(), 'Both classes required in each training fold'
    return {i: float(len(y)/(2*counts[i])) for i in (0, 1)}

def fit_head(x, y, xv, yv, seed, fixed_epochs=None):
    scaler = StandardScaler().fit(x)
    xs = scaler.transform(x).astype(np.float32)
    model = build_head(x.shape[1], seed)
    callbacks = []; validation = None
    if fixed_epochs is None:
        validation = (scaler.transform(xv).astype(np.float32), yv)
        callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                                       restore_best_weights=True)]
    history = model.fit(xs, y, validation_data=validation, epochs=fixed_epochs or MAX_EPOCHS,
                        batch_size=BATCH_SIZE, class_weight=class_weights(y),
                        callbacks=callbacks, shuffle=True, verbose=0)
    best_epoch = int(np.argmin(history.history['val_loss'])+1) if fixed_epochs is None else fixed_epochs
    return model, scaler, history.history, best_epoch

def logits(model, scaler, x):
    return model(scaler.transform(x).astype(np.float32), training=False).numpy().reshape(-1)

def metrics(y, z):
    p = expit(z); pred = (p >= 0.5).astype(int)
    return {'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
            'depressed_f1': float(f1_score(y, pred, zero_division=0)),
            'balanced_accuracy': float(balanced_accuracy_score(y, pred)),
            'auroc': float(roc_auc_score(y, p)),
            'average_precision': float(average_precision_score(y, p)),
            'brier': float(brier_score_loss(y, p)),
            'log_loss': float(log_loss(y, p, labels=[0, 1])),
            'confusion_matrix': confusion_matrix(y, pred, labels=[0, 1]).tolist()}

def save_predictions(path, ids, y, z, fold=None):
    result = pd.DataFrame({'participant_id': ids, 'label': y, 'logit': z,
                           'probability': expit(z), 'prediction': (z >= 0).astype(int)})
    if fold is not None: result['fold'] = fold
    result.to_csv(path, index=False)

def save_scaler(path, scaler):
    np.savez(path, mean=scaler.mean_, scale=scaler.scale_, var=scaler.var_)

def load_matrix(modality):
    rows = []
    for pid in manifest.participant_id:
        assert cached(pid, modality), f'Missing/stale {modality} cache for {pid}'
        with np.load(cache_file(pid, modality), allow_pickle=False) as data:
            rows.append(data['pooled'].copy())
    return np.stack(rows).astype(np.float32)

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
TEACHER_ROOT = RUN_ROOT / 'teachers' / 'frozen_encoder_v1' / run_id
TEACHER_ROOT.mkdir(parents=True, exist_ok=False)
manifest.to_csv(TEACHER_ROOT / 'train_dev_manifest.csv', index=False)
config = dict(feature_config=cache_config, cache_path=str(CACHE_ROOT), seeds=SEEDS,
              oof=DO_OOF, folds=N_FOLDS, max_epochs=MAX_EPOCHS, patience=PATIENCE,
              batch_size=BATCH_SIZE, learning_rate=3e-4, threshold=0.5,
              split_sha256={'train':train_hash, 'dev':dev_hash, 'test_ids':test_hash},
              packages={p: version(p) for p in ['tensorflow','torch','transformers','numpy','pandas','scikit-learn']},
              python=platform.python_version(), test_used_for_training_or_selection=False)
save_json(TEACHER_ROOT / 'run_config.json', config)
is_train = manifest.split.eq('train').to_numpy()
y_train = manifest.loc[is_train, 'label'].to_numpy()
y_dev = manifest.loc[~is_train, 'label'].to_numpy()
ids_train = manifest.loc[is_train, 'participant_id'].to_numpy()
ids_dev = manifest.loc[~is_train, 'participant_id'].to_numpy()
summary = []

for modality in ['audio', 'text']:
    x = load_matrix(modality); x_train, x_dev = x[is_train], x[~is_train]
    for seed in SEEDS:
        out = TEACHER_ROOT / modality / f'seed_{seed}'
        out.mkdir(parents=True, exist_ok=False)
        print(f'Training {modality} teacher; seed={seed}; input_dim={x.shape[1]}', flush=True)
        model, scaler, history, epoch = fit_head(x_train, y_train, x_dev, y_dev, seed)
        model.save(out / 'teacher.keras'); save_scaler(out / 'scaler.npz', scaler)
        pd.DataFrame(history).to_csv(out / 'history.csv', index=False)
        z_dev = logits(model, scaler, x_dev)
        z_train = logits(model, scaler, x_train)
        save_predictions(out / 'dev_predictions.csv', ids_dev, y_dev, z_dev)
        # In-sample predictions are diagnostics, NOT OOF reliability targets.
        save_predictions(out / 'train_in_sample_predictions.csv', ids_train, y_train, z_train)
        scores = metrics(y_dev, z_dev)
        save_json(out / 'dev_metrics.json', dict(scores, best_epoch=epoch))
        summary.append(dict(modality=modality, seed=seed, evaluation='dev_selection',
                            **{k:v for k,v in scores.items() if k != 'confusion_matrix'}))
        print('Development:', {k:round(v,4) for k,v in scores.items() if isinstance(v,float)}, flush=True)
        del model; gc.collect()

        if DO_OOF:
            assert np.bincount(y_train).min() >= N_FOLDS, 'Too few minority-class participants for folds'
            oof = np.full(len(y_train), np.nan); fold_ids = np.full(len(y_train), -1)
            cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
            for fold, (fit_idx, held_idx) in enumerate(cv.split(x_train, y_train)):
                inner_train, inner_val = train_test_split(fit_idx, test_size=0.2,
                    stratify=y_train[fit_idx], random_state=seed+fold)
                probe, _, inner_history, selected_epochs = fit_head(
                    x_train[inner_train], y_train[inner_train],
                    x_train[inner_val], y_train[inner_val], seed+fold)
                del probe; gc.collect()
                teacher, fold_scaler, fold_history, _ = fit_head(
                    x_train[fit_idx], y_train[fit_idx], None, None,
                    seed+1000+fold, fixed_epochs=selected_epochs)
                oof[held_idx] = logits(teacher, fold_scaler, x_train[held_idx]); fold_ids[held_idx] = fold
                fold_dir = out / 'oof' / f'fold_{fold}'; fold_dir.mkdir(parents=True, exist_ok=True)
                teacher.save(fold_dir / 'teacher.keras'); save_scaler(fold_dir / 'scaler.npz', fold_scaler)
                pd.DataFrame(inner_history).to_csv(fold_dir / 'inner_history.csv', index=False)
                pd.DataFrame(fold_history).to_csv(fold_dir / 'refit_history.csv', index=False)
                save_json(fold_dir / 'split.json', {'fit_ids':ids_train[fit_idx].tolist(),
                    'heldout_ids':ids_train[held_idx].tolist(), 'inner_train_ids':ids_train[inner_train].tolist(),
                    'inner_val_ids':ids_train[inner_val].tolist(), 'selected_epochs':selected_epochs})
                del teacher; gc.collect()
                print(f'{modality}: OOF fold {fold+1}/{N_FOLDS} complete', flush=True)
            assert np.isfinite(oof).all() and (fold_ids >= 0).all()
            save_predictions(out / 'train_oof_predictions.csv', ids_train, y_train, oof, fold_ids)
            oof_scores = metrics(y_train, oof)
            save_json(out / 'train_oof_metrics.json', oof_scores)
            summary.append(dict(modality=modality, seed=seed, evaluation='train_oof',
                                **{k:v for k,v in oof_scores.items() if k != 'confusion_matrix'}))
        pd.DataFrame(summary).to_csv(TEACHER_ROOT / 'teacher_summary.csv', index=False)

print('Saved teachers to:', TEACHER_ROOT)
display(pd.DataFrame(summary))


## 8. What to share next
Share only the displayed teacher summary and any error messages. It is safe to leave the pretrained feature caches in Drive. Reruns create new teacher run folders, preserving earlier results.

`teacher.keras` expects pooled features transformed using the associated `scaler.npz`. It does not accept raw audio/text directly. Retain `run_config.json` and feature cache configuration with checkpoints. These are **two unimodal teacher pipelines**; reliability-aware student training is the next notebook.

The full teacher uses development data for early stopping, so its development scores are not an unbiased final estimate. OOF outputs cover training participants only, and can support later reliability estimation without in-sample teacher confidence. Do not use development/test labels to fit the student. Do not claim compression from classifier parameter counts alone: frozen encoder compute and parameters must also be accounted for.

Features are derived from interview speech/text. Transcript text can itself contain explicit depression disclosures; a later cue-removal sensitivity analysis is needed before making claims about symptom inference beyond such disclosures. No publication score or novelty is guaranteed by this baseline.